Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
import pickle

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Load Second File

In [ ]:
df = pd.read_csv('/content/mental_health_combined_test.csv')
df.head()

,text,status
0,i don't understand whats wrong with me. i don'...,Anxiety
1,usually when i have anxiety just chatting with...,Anxiety
2,"well, i've had anxiety and panic syndrome for ...",Anxiety
3,"for the most minimal of things, like standing ...",Anxiety
4,i stay away from family and live with my roomm...,Anxiety


Basic Check

In [ ]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(992, 2)
Index(['text', 'status'], dtype='object')
text      0
status    0
dtype: int64


Preprocessing

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess)

In [ ]:
df['clean_text']

,clean_text
0,understand whats wrong know freak sometimes li...
1,usually anxiety chatting someone someone physi...
2,well anxiety panic syndrome years started scho...
3,minimal things like standing someone way momen...
4,stay away family live roommate literally games...
...,...
987,someone jumped building todayi wonder like
988,helpp took mg ibuprofen amp mg aspirin gt took...
989,flunked exams gonna suicideam going back home ...
990,got serious problemi considering taking life k...


Prepare Features & Labels

In [ ]:
X = df['clean_text']
y = df['status']

Split Data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Model Building (TF-IDF + SVM)

A Support Vector Machine (SVM) with a linear kernel is used for classification.
SVM is effective for high-dimensional sparse data such as TF-IDF features.

### Evaluation

The model is evaluated using accuracy, classification report, confusion matrix, sensitivity, and specificity to ensure comprehensive performance analysis across all classes.

### Deployment Readiness

The trained model and vectorizer are saved for integration into a deployment system.

Train SVM Model

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', probability=True)

svm_model.fit(X_train_tfidf, y_train)

SVC(kernel='linear', probability=True)

Evaluate Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = svm_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
import numpy as np

cm = confusion_matrix(y_test, y_pred)

sensitivity = np.diag(cm) / np.sum(cm, axis=1)

specificity = []
for i in range(len(cm)):
    tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
    fp = np.sum(cm[:, i]) - cm[i, i]
    specificity.append(tn / (tn + fp))

print("Sensitivity (per class):", sensitivity)
print("Specificity (per class):", specificity)


              precision    recall  f1-score   support

     Anxiety       0.69      0.66      0.67        44
  Depression       0.49      0.51      0.50        49
      Normal       0.69      0.73      0.70        51
    Suicidal       0.60      0.56      0.58        55

    accuracy                           0.61       199
   macro avg       0.62      0.61      0.61       199
weighted avg       0.61      0.61      0.61       199

Accuracy: 0.6130653266331658
[[29  5  5  5]
 [ 6 25  5 13]
 [ 4  7 37  3]
 [ 3 14  7 31]]
Sensitivity (per class): [0.65909091 0.51020408 0.7254902  0.56363636]
Specificity (per class): [np.float64(0.9161290322580645), np.float64(0.8266666666666667), np.float64(0.8851351351351351), np.float64(0.8541666666666666)]


In [ ]:
# 1. Convert text
X_full = vectorizer.transform(df['clean_text'])

# 2. Predict
df['predicted_label'] = svm_model.predict(X_full)

# 3. Find wrong predictions
wrong = df[df['status'] != df['predicted_label']]

# 4. Display
wrong[['text', 'status', 'predicted_label']].head()

,text,status,predicted_label
18,i had a little numbness on my left side of my ...,Anxiety,Depression
27,i think about death all the time. everything i...,Anxiety,Depression
29,for some reason i've already hated babers and ...,Anxiety,Normal
55,1. despite the buffoon serving as the american...,Anxiety,Normal
66,came across news this morning that hip hop cel...,Anxiety,Depression


Misclassified samples were analyzed to understand where the model makes errors.
The model struggles with overlapping emotional language between categories such as anxiety and depression.
Short or ambiguous text inputs also contribute to misclassification.

APPLY MODEL ON FULL DATASET

In [ ]:
X_full = vectorizer.transform(df['clean_text'])

df['predicted_label'] = svm_model.predict(X_full)

Error Analysis

In [ ]:
wrong = df[df['status'] != df['predicted_label']]

wrong[['text', 'status', 'predicted_label']].head()

,text,status,predicted_label
18,i had a little numbness on my left side of my ...,Anxiety,Depression
27,i think about death all the time. everything i...,Anxiety,Depression
29,for some reason i've already hated babers and ...,Anxiety,Normal
55,1. despite the buffoon serving as the american...,Anxiety,Normal
66,came across news this morning that hip hop cel...,Anxiety,Depression


Saving Final Output

In [ ]:
df.to_csv("final_predictions.csv", index=False)
import pickle

pickle.dump(svm_model, open("svm_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))